<a href="https://colab.research.google.com/github/hwangho-kim/Transformer_Fewshot_PdM/blob/main/%EB%B0%98%EB%8F%84%EC%B2%B4_FDC_%EC%9D%B4%EC%83%81_%ED%83%90%EC%A7%80_%EC%A0%84%EC%B2%B4_%ED%8C%8C%EC%9D%B4%ED%94%84%EB%9D%BC%EC%9D%B8_R04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
import time

# matplotlib 설정 (스타일 지정, 한글 폰트 배제)
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

print("=====================================================")
print("=== Phase 1: Realistic FDC Trace Data Generation ===")
print("=====================================================")

num_wafers = 200
seq_len = 200
num_params = 10
num_steps = 10

params_list = [
    'Pressure', 'RF_Top', 'RF_Btm', 'Gas_Cl2', 'Gas_BCl3',
    'Gas_Ar', 'Temp_Wall', 'Temp_ESC', 'VPP', 'APC_Angle'
]

step_names = [f'Step{i+1}' for i in range(num_steps)]

tensor_data = np.zeros((num_wafers, seq_len, num_params))
scenario_labels = ['Normal'] * num_wafers
np.random.seed(42)

# 센서별 반응 속도 (RC Time Constant 모사용 EMA Alpha 값)
# 1.0이면 즉각 반응, 작을수록 서서히 목표치에 도달
alpha_vals = [0.2, 0.9, 0.9, 0.15, 0.15, 0.3, 0.05, 0.02, 0.8, 0.25]

start_time = time.time()
print("Generating trace data with physical EMA characteristics and diverse anomaly scenarios...")

for w in range(num_wafers):
    # 장비의 자연 열화 (Natural Degradation) - 챔버 내 폴리머 누적으로 인한 미세한 변화
    natural_wear = (w / num_wafers)

    current_vals = np.array([5.0, 0.0, 0.0, 0.0, 0.0, 100.0, 50.0, 20.0, 0.0, 90.0]) # 초기 상태

    for t in range(seq_len):
        step_idx = t // 20

        # 스텝별 목표값 (Target Set-points)
        if step_idx == 0: # Stabilization
            target_vals = [10.0, 0.0, 0.0, 50.0, 20.0, 300.0, 50.0, 20.0, 0.0, 70.0]
        elif 1 <= step_idx <= 6: # Main Etch
            target_vals = [50.0, 1000.0, 300.0, 150.0, 50.0, 200.0, 60.0, 20.0, 800.0, 45.0]
        elif step_idx == 7: # Over Etch
            target_vals = [40.0, 800.0, 200.0, 100.0, 30.0, 250.0, 60.0, 20.0, 600.0, 55.0]
        else: # Purge & Idle
            target_vals = [5.0, 0.0, 0.0, 0.0, 0.0, 500.0, 55.0, 20.0, 0.0, 85.0]

        target_vals = np.array(target_vals)

        # 물리적 지연(RC Delay) 모사 - 이전 값에서 목표값으로 서서히 이동
        current_vals = current_vals + np.array(alpha_vals) * (target_vals - current_vals)

        # 고유 노이즈 및 자연 열화 반영
        noise = np.random.normal(0, [0.5, 5.0, 2.0, 1.0, 0.5, 2.0, 0.05, 0.01, 10.0, 0.5])
        sensor_readings = current_vals + noise

        # 특정 센서 자연 열화 반영 (온도와 압력이 미세하게 지속 상승)
        sensor_readings[0] += natural_wear * 1.5   # Pressure
        sensor_readings[6] += natural_wear * 3.0   # Temp_Wall

        # -----------------------------------------------------------------
        # 현실적인 이상 시나리오 주입 (Anomaly Injection)
        # -----------------------------------------------------------------
        # 1. Hunting (제어기 발진): Wafer 110~120, Main Etch 구간 압력/APC 요동
        if 110 <= w <= 120 and 1 <= step_idx <= 6:
            scenario_labels[w] = 'Anomaly_Hunting'
            hunting_wave = np.sin(t * 1.5) * 8.0 # 강한 진동
            sensor_readings[0] += hunting_wave       # Pressure 흔들림
            sensor_readings[9] -= hunting_wave * 0.5 # APC Angle 반대 흔들림

        # 2. Shift (영점 틀어짐): Wafer 145~155, Cl2 Gas Flow 베이스라인 이동
        if 145 <= w <= 155 and 1 <= step_idx <= 7:
            scenario_labels[w] = 'Anomaly_Shift'
            sensor_readings[3] -= 25.0 # 가스 유량이 설정치보다 항상 낮게 들어감

        # 3. Glitch (스파이크/아킹): Wafer 180~190 중 무작위 4개, 특정 스텝에서 튀어오름
        if w in [182, 185, 188, 190] and step_idx == 4:
            scenario_labels[w] = 'Anomaly_Glitch'
            if 85 <= t <= 86: # 단 2초간의 짧은 스파이크
                sensor_readings[1] -= 400.0 # RF_Top 급감 (Reflection 발생 모사)
                sensor_readings[8] += 250.0 # VPP 급증

        tensor_data[w, t, :] = sensor_readings

print(f"-> Data Generation Complete. Shape: {tensor_data.shape} ({time.time() - start_time:.2f} sec)")
print(f"-> Total Wafers: {num_wafers}")
print(f"-> Normal: {scenario_labels.count('Normal')} / Hunting: {scenario_labels.count('Anomaly_Hunting')} / Shift: {scenario_labels.count('Anomaly_Shift')} / Glitch: {scenario_labels.count('Anomaly_Glitch')}")

print("\nSaving to 'realistic_fdc_trace.csv'...")
long_format_data = []
for w in range(num_wafers):
    w_id = f'WFR-{w+1:03d}'
    label = scenario_labels[w]
    for t in range(seq_len):
        step_id = step_names[t // 20]
        timestamp = f'2026-04-01 10:{t//60:02d}:{t%60:02d}'
        for p_idx, p_name in enumerate(params_list):
            long_format_data.append([timestamp, w_id, step_id, p_name, tensor_data[w, t, p_idx], label])

df_raw = pd.DataFrame(long_format_data, columns=['TIMESTAMP', 'WAFER_ID', 'STEP_ID', 'PARA_NAME', 'VALUE', 'SCENARIO_LABEL'])
df_raw.to_csv('realistic_fdc_trace.csv', index=False)


print("\n=====================================================")
print("=== Phase 2: Data Preprocessing (Normalization)   ===")
print("=====================================================")
train_data = tensor_data[:100]

mean_val = train_data.mean(axis=(0, 1))
std_val = train_data.std(axis=(0, 1))

print("[Training Data (Wafer 1~100) Statistics - Selected Sensors]")
for i in [0, 1, 3, 6]:
    print(f" - {params_list[i]:>10}: Mean = {mean_val[i]:6.1f}, Std = {std_val[i]:5.1f}")

def normalize(data):
    return (data - mean_val) / (std_val + 1e-7)

def denormalize(data):
    return data * (std_val + 1e-7) + mean_val

tensor_data_norm = normalize(tensor_data)
train_data_norm = normalize(train_data)


print("\n=====================================================")
print("=== Phase 3: High-Performance Model Building      ===")
print("=====================================================")
class TS_MAE_Large(nn.Module):
    def __init__(self, num_features, d_model=256, nhead=8, num_layers=4):
        super().__init__()
        self.input_proj = nn.Linear(num_features, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, batch_first=True, dim_feedforward=1024, dropout=0.1
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.output_proj = nn.Linear(d_model, num_features)

    def forward(self, x, mask_ratio=0.15):
        batch_size, seq_len, _ = x.shape
        if mask_ratio > 0.0 and self.training:
            mask = torch.rand(batch_size, seq_len).to(x.device) < mask_ratio
            x_masked = x.clone()
            x_masked[mask] = 0.0
        else:
            x_masked = x

        encoded = self.input_proj(x_masked)
        out = self.transformer(encoded)
        return self.output_proj(out)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = TS_MAE_Large(num_features=num_params).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"-> Model Architecture: Transformer MAE")
print(f"-> Trainable Parameters: {total_params:,}")

if torch.cuda.device_count() > 1:
    print(f"-> Utilizing {torch.cuda.device_count()} GPUs with DataParallel!")
    model = nn.DataParallel(model)
else:
    print(f"-> Running on: {device}")

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

train_tensor = torch.FloatTensor(train_data_norm).to(device)
epochs = 80
batch_size = 16

model.train()
print("\nStarting Pre-training on Golden Wafers (1~100)...")
loss_history = []

start_train_time = time.time()
for epoch in range(epochs):
    epoch_loss = 0
    indices = torch.randperm(len(train_tensor))
    train_tensor_shuffled = train_tensor[indices]

    for i in range(0, len(train_tensor), batch_size):
        batch = train_tensor_shuffled[i:i+batch_size]
        optimizer.zero_grad()
        recon = model(batch, mask_ratio=0.20)
        loss = criterion(recon, batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    avg_loss = epoch_loss / (len(train_tensor)/batch_size)
    loss_history.append(avg_loss)
    if (epoch+1) % 20 == 0:
        print(f" - Epoch [{epoch+1:>3}/{epochs}], MSE Loss: {avg_loss:.4f}")

print(f"-> Training Complete. ({time.time() - start_train_time:.2f} sec)")

# [Visualization] Training Loss Curve
plt.figure(figsize=(8, 4))
plt.plot(loss_history, color='navy', linewidth=2)
plt.title('Self-Supervised Pre-training Loss Curve', fontsize=14)
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Reconstruction MSE Loss', fontsize=12)
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()


print("\n=====================================================")
print("=== Phase 4: Unsupervised Inference & Scatter Plot ===")
print("=====================================================")
def calculate_anomaly_scores(model, data_tensor):
    model.eval()
    with torch.no_grad():
        x = torch.FloatTensor(data_tensor).to(device)
        recon = model(x, mask_ratio=0.0)
        error_map = (x - recon) ** 2
        wafer_scores = torch.mean(error_map, dim=(1, 2)).cpu().numpy()
    return wafer_scores, error_map.cpu().numpy(), recon.cpu().numpy()

all_data_tensor = tensor_data_norm
anomaly_scores, error_maps, recon_data_norm = calculate_anomaly_scores(model, all_data_tensor)

train_scores = anomaly_scores[:100]
threshold = np.mean(train_scores) + 3 * np.std(train_scores) # 통계적 3-Sigma 임계치 적용

print(f"-> Normal Data Mean Score: {np.mean(train_scores):.4f}")
print(f"-> Normal Data Std Score:  {np.std(train_scores):.4f}")
print(f"-> Set Anomaly Threshold (Mean + 3*Std): {threshold:.4f}")

wafer_indices = np.arange(1, 201)
is_anomaly = anomaly_scores > threshold

plt.figure(figsize=(14, 6))
plt.scatter(wafer_indices[~is_anomaly], anomaly_scores[~is_anomaly],
            color='dodgerblue', alpha=0.6, label='Predicted Normal')
plt.scatter(wafer_indices[is_anomaly], anomaly_scores[is_anomaly],
            color='red', marker='x', s=80, linewidth=2, label='Predicted Anomaly')

plt.axhline(y=threshold, color='black', linestyle='--', label=f'Detection Threshold ({threshold:.4f})')

plt.title('Unsupervised Equipment Health Monitoring: Wafer Anomaly Scatter', fontsize=16)
plt.xlabel('Wafer Processing Sequence', fontsize=12)
plt.ylabel('Anomaly Score (Reconstruction Error)', fontsize=12)
plt.legend(loc='upper left')
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()


print("\n=====================================================")
print("=== Phase 5: Auto-Triggered Root Cause Analysis   ===")
print("=====================================================")
def generate_business_report(target_idx, rank):
    w_id = f'WFR-{target_idx+1:03d}'
    score = anomaly_scores[target_idx]
    e_map = error_maps[target_idx]

    print(f"\n[{rank} Rank Anomaly] RCA Report Generated for {w_id} (Score: {score:.4f})")

    plt.figure(figsize=(10, 3))
    sns.heatmap(e_map.T, cmap='Reds', cbar_kws={'label': 'Error Magnitude'})
    plt.yticks(ticks=np.arange(num_params)+0.5, labels=params_list, rotation=0, fontsize=9)
    plt.title(f'Root Cause Analysis X-Ray (Target: {w_id} - Auto Detected)', fontsize=12)
    plt.xlabel('Process Time (sec) / Step Progression', fontsize=10)
    for i in range(1, num_steps):
        plt.axvline(i * 20, color='blue', linestyle=':', alpha=0.5)
    plt.tight_layout()
    plt.show()

    actual_trace = denormalize(all_data_tensor[target_idx])
    recon_trace = denormalize(recon_data_norm[target_idx])

    fig, axes = plt.subplots(5, 2, figsize=(14, 10), sharex=True)
    axes = axes.flatten()

    for p_idx, param_name in enumerate(params_list):
        axes[p_idx].plot(actual_trace[:, p_idx], label='Actual Sensor Value', color='black', linewidth=1.2)
        axes[p_idx].plot(recon_trace[:, p_idx], label='AI Expected Value', color='red', linestyle='--', linewidth=1.2)

        diff = np.abs(actual_trace[:, p_idx] - recon_trace[:, p_idx])
        dynamic_threshold = np.std(actual_trace[:, p_idx]) * 0.5
        axes[p_idx].fill_between(range(seq_len), 0, 1, where=(diff > dynamic_threshold),
                             color='red', alpha=0.2, transform=axes[p_idx].get_xaxis_transform())

        axes[p_idx].set_title(param_name, fontsize=10, pad=3)
        if p_idx == 1:
            axes[p_idx].legend(loc='upper right', fontsize=8)
        for j in range(1, num_steps):
            axes[p_idx].axvline(j * 20, color='gray', linestyle=':', alpha=0.3)

    for i in range(8, 10):
        axes[i].set_xlabel('Process Time (sec)', fontsize=10)

    plt.suptitle(f'Detailed Sensor Analysis: Actual vs AI Expectation ({w_id})', fontsize=14, y=0.98)
    plt.tight_layout()
    plt.show()

detected_anomalies = np.where(is_anomaly)[0]
print(f"-> Total Anomalies Detected > Threshold: {len(detected_anomalies)}")

if len(detected_anomalies) > 0:
    sorted_anomalies = detected_anomalies[np.argsort(anomaly_scores[detected_anomalies])[::-1]]

    # 너무 많으면 가독성을 위해 상위 3개만 출력 (현업 시연용)
    report_limit = min(3, len(sorted_anomalies))
    print(f"-> Generating RCA reports for Top {report_limit} severe anomalies in descending order...")

    for rank, target_idx in enumerate(sorted_anomalies[:report_limit]):
        generate_business_report(target_idx, rank + 1)
else:
    print("-> No anomalies detected above the given threshold.")


print("\n=====================================================")
print("=== Phase 6: Future Trend Prediction (Prognostics) ===")
print("=====================================================")
recent_window = 100
recent_wafers = wafer_indices[-recent_window:]
recent_scores = anomaly_scores[-recent_window:]

smoothed_scores = pd.Series(recent_scores).rolling(window=5, min_periods=1).mean().values
poly_coeffs = np.polyfit(recent_wafers, smoothed_scores, 2)
poly_func = np.poly1d(poly_coeffs)

future_wafers = np.arange(num_wafers + 1, num_wafers + 51)
future_predictions = poly_func(future_wafers)

# 수식 문자열 생성 (e.g., y = 1.2e-04x^2 + -3.4e-02x + 1.5e+00)
eq_str = f"Trend Eq: $y = {poly_coeffs[0]:.2e}x^2 + {poly_coeffs[1]:.2e}x + {poly_coeffs[2]:.2e}$"
print(f"-> Extracted Polynomial Function: {eq_str}")

plt.figure(figsize=(14, 6))

plt.scatter(wafer_indices, anomaly_scores, color='lightgray', s=20, label='Actual Anomaly Scores')
plt.plot(recent_wafers, smoothed_scores, color='blue', linewidth=2, label='Smoothed Trend (Recent 100 Wafers)')
plt.plot(future_wafers, future_predictions, color='red', linestyle='--', linewidth=2.5, label='Predicted Future Trend')

plt.axhline(y=threshold, color='black', linestyle='--', label=f'Warning Threshold ({threshold:.2f})')
critical_threshold = threshold * 2.0
plt.axhline(y=critical_threshold, color='darkred', linestyle='-', label=f'Critical Failure Limit ({critical_threshold:.2f})')

# 그래프 내에 수식 텍스트 삽입
plt.text(0.02, 0.85, eq_str, transform=plt.gca().transAxes, fontsize=12,
         bbox=dict(facecolor='white', alpha=0.8, edgecolor='gray'))

plt.title('Equipment Prognostics: Future Anomaly Trend Prediction', fontsize=16)
plt.xlabel('Wafer Processing Sequence (Including Future)', fontsize=12)
plt.ylabel('Anomaly Score', fontsize=12)
plt.legend(loc='upper left')
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

print("\n=== Pipeline Execution Completed Successfully ===")